# 🎙️ Auditory-TSE 演示 Notebook\n\n本 Notebook 演示 Auditory-TSE 模型的核心功能：\n1. 模型构建与参数量分析\n2. 合成数据测试\n3. 说话人嵌入提取与可视化\n4. 目标说话人分离\n5. 分离效果对比

In [ ]:
import sys\nsys.path.insert(0, '..')\n\nimport torch\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport torchaudio\n\nfrom src.models.auditory_tse import AuditoryTSE\nfrom src.models.encoder import ConvEncoder\nfrom src.models.speaker_encoder import ECAPATDNN\n\nprint(f'PyTorch version: {torch.__version__}')\nprint(f'CUDA available: {torch.cuda.is_available()}')

## 1. 创建模型

In [ ]:
# 创建三种分离网络配置的模型\n\nmodel_conv_tasnet = AuditoryTSE(\n    encoder_channels=256,\n    speaker_embedding_dim=128,\n    separation_type='conv_tasnet',\n    bottleneck_channels=128,\n    num_tcn_blocks=6,\n    tcn_repeats=3,\n)\n\ntotal_params = sum(p.numel() for p in model_conv_tasnet.parameters())\ntrainable_params = sum(p.numel() for p in model_conv_tasnet.parameters() if p.requires_grad)\n\nprint(f'Model: Auditory-TSE (Conv-TasNet)')\nprint(f'  Total parameters:     {total_params:,}')\nprint(f'  Trainable parameters: {trainable_params:,}')\n\n# 按组件分解\ncomponents = {\n    'Encoder': model_conv_tasnet.encoder,\n    'Speaker Encoder': model_conv_tasnet.speaker_encoder,\n    'Separation': model_conv_tasnet.separation,\n    'Decoder': model_conv_tasnet.decoder,\n}\nfor name, module in components.items():\n    n = sum(p.numel() for p in module.parameters())\n    print(f'  {name:20s}: {n:>10,} params')

## 2. 合成测试数据

In [ ]:
def generate_synthetic_audio(sample_rate=16000, duration=3.0):\n    \"\"\"生成合成测试音频: 两个不同频率的说话人\"\"\"\n    t = torch.arange(int(sample_rate * duration)) / sample_rate\n    \n    # 目标说话人 (低音)\n    target = (\n        0.5 * torch.sin(2 * torch.pi * 200 * t) +\n        0.3 * torch.sin(2 * torch.pi * 400 * t) +\n        0.15 * torch.sin(2 * torch.pi * 600 * t)\n    )\n    \n    # 干扰说话人 (高音)\n    interference = (\n        0.5 * torch.sin(2 * torch.pi * 350 * t) +\n        0.3 * torch.sin(2 * torch.pi * 700 * t) +\n        0.15 * torch.sin(2 * torch.pi * 1050 * t)\n    )\n    \n    # 混合\n    mixture = target + interference\n    \n    # 注册音频: 目标说话人的另一个片段 (不同频率内容)\n    t_enr = torch.arange(int(sample_rate * 2.0)) / sample_rate\n    enrollment = (\n        0.5 * torch.sin(2 * torch.pi * 250 * t_enr) +\n        0.3 * torch.sin(2 * torch.pi * 500 * t_enr) +\n        0.15 * torch.sin(2 * torch.pi * 750 * t_enr)\n    )\n    \n    return mixture, enrollment, target, interference\n\nmixture, enrollment, target, interference = generate_synthetic_audio()\n\nprint(f'Mixture shape:      {mixture.shape}')\nprint(f'Enrollment shape:   {enrollment.shape}')\nprint(f'Target shape:       {target.shape}')\nprint(f'Interference shape: {interference.shape}')

## 3. 波形可视化

In [ ]:
def plot_waveforms(waveforms, titles, sample_rate=16000, figsize=(14, 8)):\n    \"\"\"绘制多个波形对比\"\"\"\n    n = len(waveforms)\n    fig, axes = plt.subplots(n, 1, figsize=figsize, sharex=True)\n    if n == 1:\n        axes = [axes]\n    \n    time = np.arange(len(waveforms[0])) / sample_rate\n    for ax, wf, title in zip(axes, waveforms, titles):\n        ax.plot(time[:1000], wf[:1000], linewidth=0.5)  # 前 1000 个采样点\n        ax.set_ylabel('Amplitude')\n        ax.set_title(title)\n        ax.set_ylim(-1.5, 1.5)\n        ax.grid(True, alpha=0.3)\n    \n    axes[-1].set_xlabel('Time (s)')\n    plt.tight_layout()\n    plt.show()\n\nplot_waveforms(\n    [mixture, target, interference, enrollment],\n    ['Mixture (Target + Interference)', 'Target (Ground Truth)', 'Interference', 'Enrollment (Target Reference)']\n)

## 4. 运行分离

In [ ]:
# 使用模型分离目标说话人\nmodel_conv_tasnet.eval()\n\nwith torch.no_grad():\n    output = model_conv_tasnet(\n        mixture.unsqueeze(0),      # (1, T)\n        enrollment.unsqueeze(0),   # (1, T_ref)\n        return_mask=True\n    )\n\nseparated = output['waveform'].squeeze()\nmask = output['mask'].squeeze()\nspeaker_emb = output['speaker_embedding'].squeeze()\n\nprint(f'Separated shape:     {separated.shape}')\nprint(f'Mask shape:          {mask.shape}')  # (F, T_enc)\nprint(f'Speaker embedding:   {speaker_emb.shape}')\nprint(f'Speaker emb norm:    {speaker_emb.norm():.4f}')

## 5. 分离效果对比

In [ ]:
# 对齐长度\nmin_len = min(mixture.shape[-1], separated.shape[-1], target.shape[-1])\nmixture_aligned = mixture[:min_len]\nseparated_aligned = separated[:min_len]\ntarget_aligned = target[:min_len]\n\nplot_waveforms(\n    [mixture_aligned, separated_aligned, target_aligned],\n    ['Original Mixture', 'Separated (Model Output)', 'Target (Ground Truth)']\n)

## 6. 掩码可视化

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 4))\nim = ax.imshow(mask.numpy(), aspect='auto', origin='lower', cmap='magma')\nax.set_xlabel('Time frames')\nax.set_ylabel('Feature channel')\nax.set_title('Estimated Mask (Target Speaker)')\nplt.colorbar(im, ax=ax, label='Mask value')\nplt.tight_layout()\nplt.show()

## 7. 不同分离网络对比

In [ ]:
# 创建不同分离网络的模型\nmodels = {\n    'Conv-TasNet': AuditoryTSE(\n        encoder_channels=256, speaker_embedding_dim=128,\n        separation_type='conv_tasnet'),\n    'SepFormer': AuditoryTSE(\n        encoder_channels=256, speaker_embedding_dim=128,\n        separation_type='sepformer'),\n    'Cross-Attention': AuditoryTSE(\n        encoder_channels=256, speaker_embedding_dim=128,\n        separation_type='cross_attn'),\n}\n\nfor name, model in models.items():\n    model.eval()\n    n_params = sum(p.numel() for p in model.parameters())\n    print(f'{name:20s}: {n_params:>10,} params')\n    \n    with torch.no_grad():\n        output = model(\n            mixture.unsqueeze(0),\n            enrollment.unsqueeze(0)\n        )\n    print(f'  Output shape: {output[\"waveform\"].shape}')

## 8. 说话人嵌入相似度分析

In [ ]:
# 提取多个说话人的嵌入并计算相似度\nspeaker_encoder = model_conv_tasnet.speaker_encoder\nspeaker_encoder.eval()\n\n# 模拟多个说话人的注册音频（不同频率的正弦波 = 不同的\"声纹\"）\ndef make_speaker_audio(freq, duration=2.0, sr=16000):\n    t = torch.arange(int(sr * duration)) / sr\n    audio = (\n        0.5 * torch.sin(2 * torch.pi * freq * t) +\n        0.3 * torch.sin(2 * torch.pi * freq * 2 * t) +\n        0.15 * torch.sin(2 * torch.pi * freq * 3 * t)\n    )\n    return audio\n\nspeakers = {\n    'Speaker A (200Hz)': make_speaker_audio(200),\n    'Speaker B (300Hz)': make_speaker_audio(300),\n    'Speaker C (400Hz)': make_speaker_audio(400),\n    'Speaker A - var2': make_speaker_audio(220),  # Same speaker, different content\n}\n\nwith torch.no_grad():\n    embeddings = {\n        name: speaker_encoder(audio.unsqueeze(0).unsqueeze(0))\n        for name, audio in speakers.items()\n    }\n\n# 计算余弦相似度矩阵\nnames = list(embeddings.keys())\nn = len(names)\nsim_matrix = torch.zeros(n, n)\nfor i in range(n):\n    for j in range(n):\n        sim = torch.cosine_similarity(embeddings[names[i]], embeddings[names[j]])\n        sim_matrix[i, j] = sim.item()\n\n# 可视化\nfig, ax = plt.subplots(1, 1, figsize=(8, 6))\nim = ax.imshow(sim_matrix.numpy(), cmap='RdYlGn', vmin=-1, vmax=1)\nax.set_xticks(range(n))\nax.set_yticks(range(n))\nax.set_xticklabels([n.split('(')[0].strip() for n in names], rotation=45)\nax.set_yticklabels([n.split('(')[0].strip() for n in names])\nax.set_title('Speaker Embedding Cosine Similarity')\nplt.colorbar(im, ax=ax, label='Cosine Similarity')\nplt.tight_layout()\nplt.show()

## 下一步\n\n1. **训练模型**: 使用真实数据集（LibriMix）训练\n   ```bash\n   python experiments/train.py model=auditory_tse data=librimix\n   ```\n2. **加载预训练模型**: 使用训练好的 checkpoint\n   ```python\n   pipeline = InferencePipeline.from_checkpoint('checkpoints/best.ckpt')\n   ```\n3. **自定义数据集**: 参考 `src/data/librimix.py` 实现自己的数据适配器\n4. **模型改进**: 在 `src/models/` 下添加新的分离网络